# Objetivo: compor um prato com proteína, carboidrato e vegetal com a menor quantidade de calorias

In [82]:
from dataclasses import dataclass
from enum import Enum
from itertools import cycle
import random
import itertools
from pprint import pprint

# ======== PARAMETROS ========
dias_cardapio = 5
orcamento_maximo_prato = 40
meta_calorias = 700
torelancia_caloria = 50
quantidade_ingreditens = 25

# ======== FUNCOES ========
class TipoIngrediente(Enum):
    PROTEINA = "proteina"
    CARBOIDRATO = "carboidrato"
    VEGETAL = "vegetal"


@dataclass
class Ingrediente:
    nome: str
    preco: float
    caloria: int
    tipo: TipoIngrediente


def gerar_mocks_ingredientes(quantidade: int) -> list[Ingrediente]:
    tipos = list(TipoIngrediente)
    tipos_distribuidos = cycle(tipos)

    ingredientes = []

    for i in range(quantidade):
        tipo = next(tipos_distribuidos)

        if tipo == TipoIngrediente.CARBOIDRATO:
            caloria = random.randint(200, 450)

        elif tipo == TipoIngrediente.PROTEINA:
            caloria = random.randint(150, 400)

        else:
            caloria = random.randint(20, 150)

        ingrediente = Ingrediente(
            nome=f"Ingrediente {i + 1}",
            preco=round(random.uniform(2.0, 30.0), 2),
            caloria=caloria,
            tipo=tipo
        )

        ingredientes.append(ingrediente)

    return ingredientes


def otimizar_lista_otima(ingredientes: list[Ingrediente]):
    combinacoes = itertools.product([0, 1], repeat=quantidade_ingreditens)

    combinacoes_possiveis = []

    for combinacao in combinacoes:
        if combinacao.count(1) != 3:
            continue

        calorias = 0
        preco = 0
        tipos_utilizados = []
        ingredientes_selecionados = []
        for i in range(quantidade_ingreditens):
            if combinacao[i] == 1:
                calorias += ingredientes[i].caloria
                preco += ingredientes[i].preco
                tipos_utilizados.append(ingredientes[i].tipo)
                ingredientes_selecionados.append(ingredientes[i])

        if not TipoIngrediente.VEGETAL in tipos_utilizados:
            continue

        if not TipoIngrediente.CARBOIDRATO in tipos_utilizados:
            continue

        if not TipoIngrediente.PROTEINA in tipos_utilizados:
            continue

        if not meta_calorias - torelancia_caloria <= calorias <= meta_calorias + torelancia_caloria:
            continue

        if preco > orcamento_maximo_prato:
            continue

        combinacao_info = {
            "itens": ingredientes_selecionados,
            "calorias": calorias,
            "preco": preco
        }

        combinacoes_possiveis.append(combinacao_info)

    if len(combinacoes_possiveis) < dias_cardapio:
        return None

    combinacoes_possiveis.sort(key=lambda combinacao: combinacao["preco"])

    return combinacoes_possiveis[0: dias_cardapio]



def otimizar_lista_heuristico(ingredientes: list[Ingrediente]):
    caloria_ideal_carbo = meta_calorias * 0.5
    caloria_ideal_proteina = meta_calorias * 0.4
    caloria_ideal_vegetal = meta_calorias * 0.1

    ingredientes_candidatos = {
        "carbo": [],
        "proteina": [],
        "vegetal": []
    }

    for ingrediente in ingredientes:
        if (
            ingrediente.tipo == TipoIngrediente.VEGETAL
            and caloria_ideal_vegetal - torelancia_caloria
            <= ingrediente.caloria
            <= caloria_ideal_vegetal + torelancia_caloria
        ):
            ingredientes_candidatos["vegetal"].append(ingrediente)

        if (
            ingrediente.tipo == TipoIngrediente.CARBOIDRATO
            and caloria_ideal_carbo - torelancia_caloria
            <= ingrediente.caloria
            <= caloria_ideal_carbo + torelancia_caloria
        ):
            ingredientes_candidatos["carbo"].append(ingrediente)

        if (
            ingrediente.tipo == TipoIngrediente.PROTEINA
            and caloria_ideal_proteina - torelancia_caloria
            <= ingrediente.caloria
            <= caloria_ideal_proteina + torelancia_caloria
        ):
            ingredientes_candidatos["proteina"].append(ingrediente)

    if (
        not ingredientes_candidatos["carbo"]
        or not ingredientes_candidatos["proteina"]
        or not ingredientes_candidatos["vegetal"]
    ):
        return None

    combinacoes = itertools.product(
        ingredientes_candidatos["carbo"],
        ingredientes_candidatos["proteina"],
        ingredientes_candidatos["vegetal"]
    )

    pratos_candidatos = []

    for carbo, proteina, vegetal in combinacoes:
        calorias = carbo.caloria + proteina.caloria + vegetal.caloria
        preco = carbo.preco + proteina.preco + vegetal.preco

        if not (
            meta_calorias - torelancia_caloria
            <= calorias
            <= meta_calorias + torelancia_caloria
        ):
            continue

        if preco > orcamento_maximo_prato:
            continue

        combinacao_info = {
            "itens": [carbo, proteina, vegetal],
            "calorias": calorias,
            "preco": preco
        }

        pratos_candidatos.append(combinacao_info)

    if len(pratos_candidatos) < dias_cardapio:
        return None

    pratos_candidatos.sort(
        key=lambda combinacao: combinacao["preco"]
    )

    return pratos_candidatos[:dias_cardapio]

        




        


# ======== EXECUÇÃO ========
ingredientes = gerar_mocks_ingredientes(quantidade_ingreditens)
cardapio_exaustivo = otimizar_lista_otima(ingredientes)
cardapio_heuristico = otimizar_lista_heuristico(ingredientes)


if cardapio_exaustivo and cardapio_heuristico:
    for i in range(dias_cardapio):
        print(f'Prato {i + 1}: Exaustivo: {cardapio_exaustivo[i]["preco"]} Heurístico: {cardapio_heuristico[i]["preco"]}')


Prato 1: Exaustivo: 21.27 Heurístico: 23.029999999999998
Prato 2: Exaustivo: 22.36 Heurístico: 23.52
Prato 3: Exaustivo: 23.029999999999998 Heurístico: 24.12
Prato 4: Exaustivo: 23.520000000000003 Heurístico: 24.610000000000003
Prato 5: Exaustivo: 24.12 Heurístico: 28.480000000000004
